In [3]:
## Data Preparation

import os
import re
import pandas as pd
from ase.io import read, write
from ase import Atoms
import numpy as np
import random as r
import sys
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.nn import Embedding
from torch.nn import Sequential
from torch.nn import Linear
from torch_geometric.nn import BatchNorm

csv = pd.read_csv('candidate_bonds_no_conformer.csv')

Molnums = csv['Molecule'].tolist()

#Type = csv['Predicted Type'].tolist()
#Type0 = [[typ] for typ in Type]

Atom1s = csv['Atom 1'].tolist()
Atom2s = csv['Atom 2'].tolist()

length0s = csv['Bond Length'].tolist()
#lengths = csv['Standardised Bond Length'].tolist()
projs = csv['Cos(Length)'].tolist()
bts = csv['Bond Type'].tolist()
mayers = csv['Mayer Bond'].tolist()
InRing_Bonds = csv['InRing_B'].tolist()

Elecneg_atom1 = csv['Elecneg_1'].tolist()
Elecneg_atom2 = csv['Elecneg_2'].tolist()
Ele_atom1 = csv['Ele_1'].tolist()
Ele_atom2 = csv['Ele_2'].tolist()
# C: 0, O: 1, N: 2, S: 3,
InConj_atom1 = csv['InConj_1'].tolist()
InConj_atom2 = csv['InConj_2'].tolist()
InRing_atom1 = csv['InRing_1'].tolist()
InRing_atom2 = csv['InRing_2'].tolist()
Neighb1 = csv['Neighbour1'].tolist()
Neighb2 = csv['Neighbour2'].tolist()

IsBroken_Bonds = csv['IsBroken'].tolist()


In [5]:
real_Mols = []
for num in Molnums:
    if num not in real_Mols:
        real_Mols.append(num)

real_Molnum = len(real_Mols)
#print(real_Molnum)

real_Mols = []
for i,num in enumerate(Molnums):
    if num not in real_Mols and 2.58 not in {Elecneg_atom1[i], Elecneg_atom2[i]}:
        real_Mols.append(num)

real_Molnum_skip_S = len(real_Mols)
#print(real_Molnum_skip_S)

In [6]:
#random_box = r.sample(range(real_Molnum_skip_S), k=real_Molnum_skip_S)# Skip Sulphur
random_box = r.sample(range(real_Molnum), k=real_Molnum) # Normal
print(random_box, len(random_box))

x_train = []
#x_val = []
x_test = []
y_train = []
#y_val = []
y_test = []
train_mol = []
test_mol = []
test = []#

j = 0
for turn, mol in enumerate(Molnums):
    # Skip Sulphur
    #if 2.58 in {Elecneg_atom1[turn], Elecneg_atom2[turn]}: #
    #    continue#
    
    if turn > 0 and mol != Molnums[turn-1]:
        j = j + 1
    randomnumber = random_box[j]
    if randomnumber <= real_Molnum * 0.8: # Normal
    #if randomnumber <= real_Molnum_skip_S * 0.8: # Skip Sulphur
        train_mol.append([mol, [Atom1s[turn], Atom2s[turn]]])
        #x_train.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], Ele_atom1[turn],\
        #               Ele_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        x_train.append(np.array([length0s[turn], bts[turn], projs[turn], mayers[turn], InRing_Bonds[turn], Ele_atom1[turn],\
                       Ele_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        y_train.append(np.array(IsBroken_Bonds[turn]))
        if randomnumber not in test:
            test.append(randomnumber)
    else:
        test_mol.append([mol, [Atom1s[turn], Atom2s[turn]]])
        #x_test.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], Ele_atom1[turn],\
        #               Ele_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        x_test.append(np.array([length0s[turn], bts[turn], projs[turn], mayers[turn], InRing_Bonds[turn], Ele_atom1[turn],\
                       Ele_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        y_test.append(np.array(IsBroken_Bonds[turn]))
        if randomnumber not in test:
            test.append(randomnumber)

print(len(x_train))
print(len(x_test))

[425, 21, 91, 475, 67, 649, 147, 146, 460, 654, 32, 230, 48, 58, 409, 648, 481, 165, 658, 160, 295, 619, 626, 659, 74, 466, 151, 593, 371, 582, 476, 509, 95, 473, 31, 218, 433, 187, 339, 22, 121, 38, 529, 26, 480, 540, 342, 300, 657, 175, 422, 284, 155, 61, 617, 234, 597, 168, 596, 482, 562, 641, 307, 163, 545, 415, 266, 220, 398, 387, 98, 396, 592, 235, 553, 123, 558, 34, 652, 205, 308, 421, 259, 615, 36, 585, 572, 132, 166, 497, 439, 431, 265, 355, 397, 623, 77, 116, 316, 349, 520, 62, 281, 174, 10, 605, 199, 622, 63, 493, 530, 632, 33, 50, 271, 119, 502, 202, 441, 42, 343, 551, 73, 634, 373, 479, 614, 432, 96, 287, 503, 534, 240, 331, 209, 326, 357, 539, 337, 35, 45, 315, 210, 17, 633, 526, 256, 5, 242, 82, 498, 144, 294, 263, 577, 305, 524, 84, 213, 59, 103, 567, 93, 118, 262, 462, 19, 23, 527, 500, 583, 340, 157, 536, 310, 389, 112, 224, 99, 221, 434, 169, 508, 380, 285, 603, 638, 176, 206, 410, 565, 299, 573, 251, 298, 484, 40, 257, 49, 428, 639, 571, 212, 363, 106, 177, 350, 289

In [57]:
## Cross Validation Mode

x_train = []
#x_val = []
x_test = []
y_train = []
#y_val = []
y_test = []
train_mol = []
test_mol = []
test = []#

j = 0
for turn, mol in enumerate(Molnums):
    #if 2.58 in {Elecneg_atom1[turn], Elecneg_atom2[turn]}: #
    #    continue#
    
    if turn > 0 and mol != Molnums[turn-1]:
        j = j + 1
    randomnumber = random_box[j]
    
    #if randomnumber <= real_Molnum * 0.8: # Skip Sulphur
    if randomnumber < real_Molnum * 0.0 or randomnumber >= real_Molnum * 0.2: # Skip Sulphur
        train_mol.append([mol, [Atom1s[turn], Atom2s[turn]]])
        #x_train.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], Elecneg_atom1[turn],\
        #               Elecneg_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        x_train.append(np.array([length0s[turn], bts[turn], projs[turn], mayers[turn], InRing_Bonds[turn], Elecneg_atom1[turn],\
                       Elecneg_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        y_train.append(np.array(IsBroken_Bonds[turn]))
        if randomnumber not in test:
            test.append(randomnumber)
    else:
        test_mol.append([mol, [Atom1s[turn], Atom2s[turn]]])
        #x_test.append(np.array([length0s[turn], bts[turn], mayers[turn], InRing_Bonds[turn], Elecneg_atom1[turn],\
        #              Elecneg_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        x_test.append(np.array([length0s[turn], bts[turn], projs[turn], mayers[turn], InRing_Bonds[turn], Elecneg_atom1[turn],\
                      Elecneg_atom2[turn], InConj_atom1[turn], InConj_atom2[turn], Neighb1[turn], Neighb2[turn]]))
        y_test.append(np.array(IsBroken_Bonds[turn]))
        if randomnumber not in test:
            test.append(randomnumber)

print(len(x_train))
print(len(x_test))

5108
1336


In [58]:
y_train = np.array(y_train)
y_test = np.array(y_test)

In [59]:
all_box_x = []
box_x = []
all_box_y = []
box_y = []
all_train_mol_details = []
train_mol_details = []

# Assemble
for i, mol in enumerate(train_mol):
    if i >= 1 and train_mol[i][0] != train_mol[i-1][0]:
        all_box_x.append(box_x)
        all_train_mol_details.append(train_mol_details)
        all_box_y.append(box_y)
        box_x = []
        box_y = []
        train_mol_details = []
    elif i == len(train_mol) -1:
        all_box_x.append(box_x)
        all_train_mol_details.append(train_mol_details)
        all_box_y.append(box_y)
    
    train_mol_details.append(train_mol[i])
    box_x.append(x_train[i])
    box_y.append(y_train[i])

total = len(all_box_x)
print('Total Train Number:', len(all_box_x))

Total Train Number: 528


In [25]:
all_box_x[0]

[array([1.53475155, 0.        , 1.12578704, 0.85522737, 0.        ,
        2.55      , 2.55      , 0.        , 0.        , 3.        ,
        4.        ]),
 array([1.52140198, 0.        , 1.0114858 , 1.07484272, 0.        ,
        2.55      , 2.55      , 0.        , 0.        , 2.        ,
        2.        ]),
 array([1.51602731, 0.        , 0.97942653, 1.04655467, 0.        ,
        2.55      , 2.55      , 0.        , 0.        , 2.        ,
        2.        ]),
 array([1.51492602, 0.        , 1.09754041, 0.93185244, 0.        ,
        2.55      , 2.55      , 0.        , 0.        , 3.        ,
        3.        ]),
 array([1.50850872, 0.        , 1.27415313, 1.04522239, 0.        ,
        2.55      , 2.55      , 0.        , 0.        , 3.        ,
        1.        ]),
 array([1.50824772, 0.        , 1.27354819, 1.04529736, 0.        ,
        2.55      , 2.55      , 0.        , 0.        , 1.        ,
        3.        ]),
 array([1.43651172, 1.        , 0.9800529 , 0.884215

In [10]:
all_train_mol_details[0]

[['1', [20, 21]],
 ['1', [17, 16]],
 ['1', [4, 11]],
 ['1', [0, 2]],
 ['1', [35, 36]],
 ['1', [31, 30]],
 ['1', [15, 20]],
 ['1', [16, 1]],
 ['1', [21, 27]],
 ['1', [24, 17]],
 ['1', [30, 24]],
 ['1', [27, 35]],
 ['1', [11, 15]],
 ['1', [1, 0]],
 ['1', [2, 4]]]

In [11]:
all_box_y[0]

[np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(0),
 np.int64(1)]

In [76]:
## Data Examine

pre_edge1 = []
pre_edge2 = []
attract = []
labels = []
x = []
atomnum = []

box_x = all_box_x[0]
train_mol_details = all_train_mol_details[0]

for i, index in enumerate(box_x):
    
    for l, j in enumerate(train_mol_details[i][1]):
        if j not in atomnum and l == 0:
            atomnum.append(j)
            x_atom = []
            x_atom.append(box_x[i][2])
            x_atom.append(box_x[i][4])
            x_atom.append(box_x[i][6])
            x.append(x_atom)
        elif j not in atomnum and l == 1:
            atomnum.append(j)
            x_atom = []
            x_atom.append(box_x[i][3])
            x_atom.append(box_x[i][5])
            x_atom.append(box_x[i][5])
            x.append(x_atom)
    pre_edge1.append([*train_mol_details[i][1]][0])
    pre_edge1.append([*train_mol_details[i][1]][1])
    train_mol_details[i][1].reverse()
    reve = [*train_mol_details[i][1]]
    train_mol_details[i][1].reverse()
    pre_edge2.append(reve[0])
    pre_edge2.append(reve[1])
    attract0 = []
    attract0.append(box_x[i][0])
    attract0.append(box_x[i][1])
    attract.append(attract0)
    attract0 = []
    attract0.append(box_x[i][0])
    attract0.append(box_x[i][1])
    attract.append(attract0)
    labels.append(box_y[i])
    labels.append(box_y[i])

print(pre_edge1, pre_edge2)
print(atomnum)
print(x)
print(attract)
print(labels)

IndexError: list index out of range

In [12]:
print(len(random_box))

661


In [60]:
def ring_index_conform(x):
    if x == 0:
        return 0
    else:
        return x - 2

## Make Dataset

Dataset = []
train_Dataset = []
#val_Dataset = []
#test_Dataset = []

xt = 1

for turn, box_x in enumerate(all_box_x):
    pre_edge1 = []
    pre_edge2 = []
    attract = []
    labels = []
    x = []
    atomnum = []

    box_x = all_box_x[turn]
    box_y = all_box_y[turn]
    train_mol_details = all_train_mol_details[turn]
    print('Molecule {0}'.format(train_mol_details[0][0]))

    for i, index in enumerate(box_x):
        for l, j in enumerate(train_mol_details[i][1]):
            if j not in atomnum and l == 0:
                atomnum.append(j)
                x_atom = []
                x_atom.append(box_x[i][4+xt]) # 3
                x_atom.append(box_x[i][6+xt]) # 5
                x_atom.append(box_x[i][8+xt]) # 7
                x.append(x_atom)
            elif j not in atomnum and l == 1:
                atomnum.append(j)
                x_atom = []
                x_atom.append(box_x[i][5+xt]) # 4
                x_atom.append(box_x[i][7+xt]) # 6
                x_atom.append(box_x[i][9+xt]) # 8
                x.append(x_atom)
        pre_edge1.append([*train_mol_details[i][1]][0])
        pre_edge1.append([*train_mol_details[i][1]][1])
        train_mol_details[i][1].reverse()
        reve = [*train_mol_details[i][1]]
        train_mol_details[i][1].reverse()
        pre_edge2.append(reve[0])
        pre_edge2.append(reve[1])
        attract0 = []
        attract0.append(box_x[i][0])
        attract0.append(box_x[i][1])
        attract0.append(box_x[i][2])
        attract0.append(box_x[i][3]) # Delete for 4
        attract0.append(ring_index_conform(box_x[i][4]))#attract0.append(ring_index_conform(box_x[i][3]))
        attract.append(attract0)
        attract0 = []
        attract0.append(box_x[i][0])
        attract0.append(box_x[i][1])
        attract0.append(box_x[i][2])
        attract0.append(box_x[i][3]) # Delete for 4
        attract0.append(ring_index_conform(box_x[i][4]))#attract0.append(ring_index_conform(box_x[i][3]))
        attract.append(attract0)
        labels.append(box_y[i])
        labels.append(box_y[i])

    #print([pre_edge1, pre_edge2])
    #print(atomnum)
    
    # Remap atom numbers
    atom_to_idx = {atom: i for i, atom in enumerate(atomnum)}
    src_original = train_mol_details[i][1][0]
    dst_original = train_mol_details[i][1][1]
    pre_edge1_modified = [atom_to_idx[i] for i in pre_edge1]
    pre_edge2_modified = [atom_to_idx[i] for i in pre_edge2]
    
    print([pre_edge1_modified, pre_edge2_modified])
    print(atom_to_idx)
    
    print(x)
    print(attract)
    print(labels)

    # data features (X)
    X = torch.tensor(x, dtype=torch.float)

    # nodes / links
    edge_index = torch.tensor([pre_edge1_modified, pre_edge2_modified], dtype=torch.long)
    edge_attr = torch.tensor(attract, dtype=torch.float)
    edge_labels = torch.tensor(labels, dtype=torch.long)

    # build data
    data = Data(x=X, edge_index=edge_index, edge_attr=edge_attr, edge_labels=edge_labels)
    print(data)
    # Data(x=[4,2], edge=[2, 5], y=[4])
    print('Isolated nodes: {0}'.format(data.has_isolated_nodes()))
    print('Loops: {0}'.format(data.has_self_loops()))
    print('Undirected: {0}'.format(data.is_undirected()))
    print()
    
    randomnumber = random_box[turn]
    train_Dataset.append(data)

#torch.save(Dataset, './model.pth')
torch.save(train_Dataset, './model_train.pth')
#torch.save(val_Dataset, './model_val.pth')
#torch.save(test_Dataset, './model_test.pth')

#print('All:       ', len(Dataset))
print('Training:  ', len(train_Dataset))
#print('Validation:', len(val_Dataset))
#print('Testing:   ', len(test_Dataset))

Molecule 1
[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 3, 13, 1, 14, 15, 2, 11, 15, 14, 8, 5, 12, 13, 6, 7, 4], [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10, 0, 12, 13, 3, 14, 1, 2, 15, 15, 11, 8, 14, 12, 5, 6, 13, 4, 7]]
{20: 0, 21: 1, 17: 2, 16: 3, 4: 4, 11: 5, 0: 6, 2: 7, 35: 8, 36: 9, 31: 10, 30: 11, 15: 12, 1: 13, 27: 14, 24: 15}
[[np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(1.0)], [np.float64(2.55), np.float64(0.0), np.float64(1.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(3.44),

In [61]:
all_box_x = []
box_x = []
all_box_y = []
box_y = []
all_test_mol_details = []
test_mol_details = []

# Assemble
for i, mol in enumerate(test_mol):
    if i >= 1 and test_mol[i][0] != test_mol[i-1][0]:
        all_box_x.append(box_x)
        all_test_mol_details.append(test_mol_details)
        all_box_y.append(box_y)
        box_x = []
        box_y = []
        test_mol_details = []
    elif i == len(train_mol) -1:
        all_box_x.append(box_x)
        all_test_mol_details.append(test_mol_details)
        all_box_y.append(box_y)
    
    test_mol_details.append(test_mol[i])
    box_x.append(x_test[i])
    box_y.append(y_test[i])

total = len(all_box_x)
print('Total Test Number:', len(all_box_x))

Total Test Number: 132


In [62]:
all_test_mol_details[0]

[['3', [0, 2]],
 ['3', [20, 21]],
 ['3', [17, 16]],
 ['3', [4, 11]],
 ['3', [35, 36]],
 ['3', [31, 30]],
 ['3', [16, 1]],
 ['3', [15, 20]],
 ['3', [21, 27]],
 ['3', [24, 17]],
 ['3', [30, 24]],
 ['3', [27, 35]],
 ['3', [11, 15]],
 ['3', [1, 0]],
 ['3', [2, 4]]]

In [63]:
## Make Dataset

Dataset = []
#train_Dataset = []
#val_Dataset = []
test_Dataset = []

for turn, box_x in enumerate(all_box_x):
    pre_edge1 = []
    pre_edge2 = []
    attract = []
    labels = []
    x = []
    atomnum = []

    box_x = all_box_x[turn]
    box_y = all_box_y[turn]
    test_mol_details = all_test_mol_details[turn]
    print('Molecule {0}'.format(test_mol_details[0][0]))

    for i, index in enumerate(box_x):
        for l, j in enumerate(test_mol_details[i][1]):
            if j not in atomnum and l == 0:
                atomnum.append(j)
                x_atom = []
                x_atom.append(box_x[i][4+xt]) # 3
                x_atom.append(box_x[i][6+xt]) # 5
                x_atom.append(box_x[i][8+xt]) # 7
                x.append(x_atom)
            elif j not in atomnum and l == 1:
                atomnum.append(j)
                x_atom = []
                x_atom.append(box_x[i][5+xt]) # 4
                x_atom.append(box_x[i][7+xt]) # 6
                x_atom.append(box_x[i][9+xt]) # 8
                x.append(x_atom)
        pre_edge1.append([*test_mol_details[i][1]][0])
        pre_edge1.append([*test_mol_details[i][1]][1])
        test_mol_details[i][1].reverse()
        reve = [*test_mol_details[i][1]]
        test_mol_details[i][1].reverse()
        pre_edge2.append(reve[0])
        pre_edge2.append(reve[1])
        attract0 = []
        attract0.append(box_x[i][0])
        attract0.append(box_x[i][1])
        attract0.append(box_x[i][2])
        attract0.append(box_x[i][3]) # Delete for 4
        attract0.append(ring_index_conform(box_x[i][4]))#attract0.append(ring_index_conform(box_x[i][3]))
        attract.append(attract0)
        attract0 = []
        attract0.append(box_x[i][0])
        attract0.append(box_x[i][1])
        attract0.append(box_x[i][2])
        attract0.append(box_x[i][3]) # Delete for 4
        attract0.append(ring_index_conform(box_x[i][4]))#attract0.append(ring_index_conform(box_x[i][3]))
        attract.append(attract0)
        labels.append(box_y[i])
        labels.append(box_y[i])
    
    # Remap atom numbers
    atom_to_idx = {atom: i for i, atom in enumerate(atomnum)}
    src_original = test_mol_details[i][1][0]
    dst_original = test_mol_details[i][1][1]
    pre_edge1_modified = [atom_to_idx[i] for i in pre_edge1]
    pre_edge2_modified = [atom_to_idx[i] for i in pre_edge2]
    
    print([pre_edge1_modified, pre_edge2_modified])
    print(atom_to_idx)
    
    print(x)
    print(attract)
    print(labels)

    # data features (X)
    X = torch.tensor(x, dtype=torch.float)

    # nodes / links
    edge_index = torch.tensor([pre_edge1_modified, pre_edge2_modified], dtype=torch.long)
    edge_attr = torch.tensor(attract, dtype=torch.float)
    edge_labels = torch.tensor(labels, dtype=torch.long)

    # build data
    data = Data(x=X, edge_index=edge_index, edge_attr=edge_attr, edge_labels=edge_labels)
    print(data)
    # Data(x=[4,2], edge=[2, 5], y=[4])
    print('Isolated nodes: {0}'.format(data.has_isolated_nodes()))
    print('Loops: {0}'.format(data.has_self_loops()))
    print('Undirected: {0}'.format(data.is_undirected()))
    print()
    
    randomnumber = random_box[turn]
    test_Dataset.append(data)

#torch.save(Dataset, './model.pth')
#torch.save(train_Dataset, './model_train.pth')
#torch.save(val_Dataset, './model_val.pth')
torch.save(test_Dataset, './model_test.pth')

#print('All:       ', len(Dataset))
#print('Training:  ', len(train_Dataset))
#print('Validation:', len(val_Dataset))
print('Testing:   ', len(test_Dataset))

Molecule 3
[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 5, 12, 13, 2, 3, 14, 15, 4, 11, 15, 14, 8, 7, 13, 12, 0, 1, 6], [1, 0, 3, 2, 5, 4, 7, 6, 9, 8, 11, 10, 12, 5, 2, 13, 14, 3, 4, 15, 15, 11, 8, 14, 13, 7, 0, 12, 6, 1]]
{0: 0, 2: 1, 20: 2, 21: 3, 17: 4, 16: 5, 4: 6, 11: 7, 35: 8, 36: 9, 31: 10, 30: 11, 1: 12, 15: 13, 27: 14, 24: 15}
[[np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(4.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(2.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(2.55), np.float64(0.0), np.float64(1.0)], [np.float64(2.55), np.float64(0.0), np.float64(1.0)], [np.float64(2.55), np.float64(0.0), np.float64(3.0)], [np.float64(3.44),

In [ ]:
from torch_geometric.nn import BatchNorm
from torch.nn import Embedding
from torch.nn import Sequential

class ReactionSitePredictor(torch.nn.Module):
    def __init__(self, node_features, edge_attr_dim, edge_classes=2):
        
        # Nodes
        
        self.node_embedding_layers = torch.nn.ModuleDict()
        for feat_name, vocab_size in node_categorical_dim.items():
            embed_dim = min(50, max(8, vocab_size // 2 + 1))
            self.node_embedding_layers[feat_name] = torch.nn.Embedding(
                vocab_size, embed_dim, padding_idx=0
            )
        
        # Electronegativity: 0
        self.embedding_inconj = Embedding(2, 1)       # InConj: 1
        self.embedding_neighb = Embedding(5, 2)       # Neighbour: 2
        self.node_embedding_number = 1 + 2
        
        node1_nn1 = Sequential(Linear(self.node_embedding_number, 32), ReLU(),\
                               Linear(32, 16))
        node_nn1 = Sequential(Linear(17, 16), ReLU(),\
                              Linear(16, 16))

        self.node1_conv1 = GCNConv(node1_nn1)
        self.node1_bn1 = torch.nn.BatchNorm1d(16)
        self.node_conv1 = GCNConv(node_nn1)
        self.node_bn1 = torch.nn.BatchNorm1d(16)
        
        # Edge
        
        self.edge1_linear = Sequential(Linear(self.edge_embedding_number, 16), ReLU(),\
                                       Linear(16, 16))
        self.edge2_linear = Sequential(Linear(18, 16), ReLU(),\
                                       Linear(16, 16))
        
        
        # Bond Length: 0
        self.embedding_bondtype = Embedding(10, 4)    # Bond Type: 1
        # Mayer Bond Level: 2
        self.edge_embedding_number = 4
        
        
        
        super(ReactionSitePredictor, self).__init__()
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.bn1 = BatchNorm(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = BatchNorm(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        
        self.edge_linear = Sequential(Linear(self.edge_embedding_number, 32), ReLU(),\
                                      Linear(32, 16))
        
        
        self.edge_classifier = torch.nn.Sequential(
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(16, edge_classes)
        )
        
        self.dropout = torch.nn.Dropout(0.1)
        
    #def forward(self, data):
    def forward(self, x, edge_index, edge_attr, batch):
        #x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        # Nodes
        
        node1 = self.embedding_inconj(x[:, 0].long()) # InConj: 0
        node2 = x[:, 1:2]                           v # Electronegativity: 1
        node3 = self.embedding_neighb(x[:, 2].long()) # Neighbour: 2
        
        node1 = torch.cat([node1, node3], dim=1)
        x1 = self.node1_conv1(node1, edge_index)
        x1 = self.dropout(x1)
        x1 = F.relu(x1)
        x1 = self.node1_bn1(x1)
        
        x = torch.cat([x1, node2], dim=1)

        x = self.node_conv1(x, edge_index)
        x = self.dropout(x)
        x = F.relu(x)
        x = self.node_bn1(x)
        
        # Edges
        src_node = edge_index[0]
        dst_node = edge_index[1]
            
        src_feat = x[src_node]
        dst_feat = x[dst_node]
        
        edge1 = edge_attr[:, 0:1]                               # Bond Length: 0
        edge2 = self.embedding_bondtype(edge_attr[:, 1].long()) # Bond Type: 1
        edge3 = edge_attr[:, 2:3]                               # Mayer Bond Level: 2
        edge_attr0 = torch.cat([edge2], dim=1)
        
        edge_attr0 = self.edge_linear(edge_attr0)
        edge_attr0 = torch.cat([edge1, edge3, edge_attr0], dim=1)
        edge_attr0 = self.edge_linear2(edge_attr0)
        
        edge_batch = batch[edge_index[0]]
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)  # Final node
        

            
        edge_features = torch.cat([src_feat, dst_feat, edge_attr], dim=-1)
        
        edge_predictions = self.edge_classifier(edge_features)
        
        return edge_predictions

In [ ]:
class ReactionSitePredictor(torch.nn.Module):
    def __init__(self, node_features, hidden_dim, edge_attr_dim, edge_classes=2):
        super(ReactionSitePredictor, self).__init__()
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        
        self.edge_classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim * 2 + edge_attr_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_dim, edge_classes)
        )
        
    #def forward(self, data):
    def forward(self, x, edge_index, edge_attr, batch):
        #x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)  # Final node
        
        src_node = edge_index[0]
        dst_node = edge_index[1]
            
        src_feat = x[src_node]
        dst_feat = x[dst_node]
            
        edge_features = torch.cat([src_feat, dst_feat, edge_attr], dim=-1)
        
        edge_predictions = self.edge_classifier(edge_features)
        
        return edge_predictions

In [64]:
class ReactionSitePredictor(torch.nn.Module):
    def __init__(self, hidden_dim, edge_classes=2):
        super(ReactionSitePredictor, self).__init__()
        
        ## Nodes
        # Electronegativity: 0a
        self.embedding_ele = Embedding(4, 3)       # Element: 0b
        self.embedding_inconj = Embedding(2, 1)       # InConj: 1
        self.embedding_neighb = Embedding(5, 4)       # Neighbour: 2
        self.node_embedding_number = 3 + 1 + 4
        
        ## Edges
        # Bond Length: 0
        self.embedding_bondtype = Embedding(10, 8)    # Bond Type: 1
        # Mayer Bond Level: 2
        # Projection: 3 # Deleta for 4
        self.embedding_inring = Embedding(6, 4)       # Bond In Ring: 4 # 3
        self.edge_embedding_number = 1 + 8 + 1 + 1 + 4 # 1 + 8 + 1 + 4
        
        self.conv1 = GCNConv(self.node_embedding_number, hidden_dim)
        self.bn1 = BatchNorm(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2 = BatchNorm(hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        
        self.edge_classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim * 2 + self.edge_embedding_number, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_dim, edge_classes)
        )
        
    #def forward(self, data):
    def forward(self, x, edge_index, edge_attr, batch):
        #x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        # Ensure inputs are Integers (Long) and remove extra dimensions if [N, 1]
        #x = x.long().squeeze()
        #edge_attr = edge_attr.long().squeeze()
        
        #node1 = x[:, 0:1]                             # Electronegativity: 0a
        node1 = self.embedding_ele(x[:, 0].long())    # Element: 0b
        node2 = self.embedding_inconj(x[:, 1].long()) # InConj: 1
        node3 = self.embedding_neighb(x[:, 2].long()) # Neighbour: 2
        node_all = torch.cat([node1, node2, node3], dim=-1)
        
        x_ = self.conv1(node_all, edge_index)
        x_ = F.relu(self.bn1(x_))
        x_ = self.conv2(x_, edge_index)
        x_ = F.relu(self.bn2(x_))
        x_ = self.conv3(x_, edge_index)  # Final node
        
        edge1 = edge_attr[:, 0:1]                               # Bond Length: 0
        edge2 = self.embedding_bondtype(edge_attr[:, 1].long()) # Bond Type: 1
        edge3 = edge_attr[:, 2:3]                               # Mayer Bond Level: 2
        edge4 = edge_attr[:, 3:4]                               # Projection: 3  # Delete for 4
        edge5 = self.embedding_inring(edge_attr[:, 4].long())   # Bond In Ring: 4 # Bond In Ring: 3
        edge_all = torch.cat([edge1, edge2, edge3, edge4, edge5], dim=-1)
        
        src_node = edge_index[0]
        dst_node = edge_index[1]
            
        src_feat = x_[src_node]
        dst_feat = x_[dst_node]
            
        edge_features = torch.cat([src_feat, dst_feat, edge_all], dim=-1)
        
        edge_predictions = self.edge_classifier(edge_features)
        
        return edge_predictions

In [65]:
train_loader = DataLoader(train_Dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_Dataset, batch_size=32, shuffle=False)

In [72]:
def train_model():
    stop_signal = 0
    
    model = ReactionSitePredictor(hidden_dim=32, edge_classes=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(100):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            
            #edge_predictions = model(batch)
            edge_predictions = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            
            loss = criterion(edge_predictions, batch.edge_labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        if epoch % 5 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss/len(train_loader):.4f}')
        
        if total_loss / len(train_loader) <= 0.05:
            stop_signal = stop_signal + 1
        else:
            stop_signal = 0
            
        if stop_signal == 5:
            break
    
    return model

In [73]:
#def predict_reaction_sites(model, molecule):
def predict_reaction_sites(model, x, edge_index, edge_attr, batch):
    model.eval()
    with torch.no_grad():
        #edge_predictions = model(molecule)
        edge_predictions = model(x, edge_index, edge_attr, batch)
        edge_probs = F.softmax(edge_predictions, dim=-1)
        predicted_classes = torch.argmax(edge_probs, dim=1)
        
        return edge_probs, predicted_classes

trained_model = train_model()


Epoch 0, Loss: 0.2541
Epoch 5, Loss: 0.0960
Epoch 10, Loss: 0.0720
Epoch 15, Loss: 0.0805
Epoch 20, Loss: 0.0935
Epoch 25, Loss: 0.0613
Epoch 30, Loss: 0.0454
Epoch 35, Loss: 0.0703
Epoch 40, Loss: 0.0454


In [31]:
# Bonds

all_sets = 0
correct_sets = 0

r_r = 0
r_w = 0
r_r_wrong = 0
w_r = 0
w_w = 0

for i in range(len(all_test_mol_details)):
    #edge_probs, predicted_classes = predict_reaction_sites(trained_model, test_Dataset[i])
    edge_probs, predicted_classes = predict_reaction_sites(trained_model, test_Dataset[i].x,\
                                                           test_Dataset[i].edge_index, test_Dataset[i].edge_attr,\
                                                           test_Dataset[i].batch)
    
    print(f"Bonds: {[test_Dataset[i].edge_attr.numpy()]}")
    
    print(f"Molecule {all_test_mol_details[i][0][0]}:")
    print(f"Edges: {test_Dataset[i].edge_index.size(1)}")
    print(f"Possibility: {edge_probs[:, 1].numpy()}")
    print(f"Predicted: {predicted_classes.numpy()}")
    print(f"Target: {test_Dataset[i].edge_labels.numpy()}")
    
    
    ox = ([*predicted_classes] == [*test_Dataset[i].edge_labels])
    print(ox)
    predicted_classes_list = [*predicted_classes]
    test_Dataset_list = [*test_Dataset[i].edge_labels]
    for j,res in enumerate(predicted_classes_list):
        all_sets = all_sets + 1
        if predicted_classes_list[j] == test_Dataset_list[j]:
            correct_sets = correct_sets + 1
            if predicted_classes_list[j] == 1:
                r_r = r_r + 1
            else:
                r_w = r_w + 1
        else:
            if predicted_classes_list[j] == 1:
                w_w = w_w + 1
            else:
                w_r = w_r + 1
        
    reaction_edges = torch.where(predicted_classes == 1)[0]
    print(f"Bond: {reaction_edges.numpy()}")

    print("\nAll bonds:")
    for edge_idx in reaction_edges:
        src_atom = test_Dataset[i].edge_index[0, edge_idx].item()
        dst_atom = test_Dataset[i].edge_index[1, edge_idx].item()
        probability = edge_probs[edge_idx, 1].item()
        print(f"  Bond {edge_idx}: Atom {src_atom} - Atom {dst_atom}, Possibility: {probability:.3f}")
    
    print()

print('All datas:', all_sets)
print('Correct:', correct_sets)
print('Accuracy:', correct_sets / all_sets)
print()
print()
print('{0} {1}'.format(r_r,  w_r))
print('{0} {1}'.format(w_w, r_w))

Bonds: [array([[1.5142534 , 0.        , 0.9059073 , 0.        ],
       [1.5142534 , 0.        , 0.9059073 , 0.        ],
       [1.5140226 , 0.        , 0.91404366, 0.        ],
       [1.5140226 , 0.        , 0.91404366, 0.        ],
       [1.4309621 , 1.        , 0.91375774, 0.        ],
       [1.4309621 , 1.        , 0.91375774, 0.        ],
       [1.4294038 , 1.        , 0.91870606, 0.        ],
       [1.4294038 , 1.        , 0.91870606, 0.        ],
       [1.3444238 , 1.        , 1.2045226 , 0.        ],
       [1.3444238 , 1.        , 1.2045226 , 0.        ],
       [1.3405601 , 1.        , 1.2010725 , 0.        ],
       [1.3405601 , 1.        , 1.2010725 , 0.        ],
       [1.561778  , 0.        , 1.1068531 , 2.        ],
       [1.561778  , 0.        , 1.1068531 , 2.        ]], dtype=float32)]
Molecule 7:
Edges: 14
Possibility: [2.8632372e-04 5.1115215e-04 5.0359225e-04 2.8276152e-04 2.6507552e-10
 2.4329153e-10 2.3907462e-10 2.6048105e-10 5.1480118e-08 1.5127782e-08


In [74]:
# Molecule

all_sets = 0
correct_sets = 0

r_r = 0
r_w = 0
r_r_wrong = 0
w_r = 0
w_w = 0

for i in range(len(all_test_mol_details)):
    #edge_probs, predicted_classes = predict_reaction_sites(trained_model, test_Dataset[i])
    edge_probs, predicted_classes = predict_reaction_sites(trained_model, test_Dataset[i].x,\
                                                           test_Dataset[i].edge_index, test_Dataset[i].edge_attr,\
                                                           test_Dataset[i].batch)
    
    print(f"Bonds: {[test_Dataset[i].edge_attr.numpy()]}")
    
    print(f"Molecule {all_test_mol_details[i][0][0]}:")
    print(f"Edges: {test_Dataset[i].edge_index.size(1)}")
    print(f"Possibility: {edge_probs[:, 1].numpy()}")
    print(f"Predicted: {predicted_classes.numpy()}")
    print(f"Target: {test_Dataset[i].edge_labels.numpy()}")
    ox = ([*predicted_classes] == [*test_Dataset[i].edge_labels])
    print(ox)
    all_sets = all_sets + 1
    if ox == True:
        correct_sets = correct_sets + 1
        if 1 in [*test_Dataset[i].edge_labels]:
            r_r += 1
        else:
            r_w += 1
    else:
        if 1 in [*test_Dataset[i].edge_labels] and 1 in [*train_Dataset[i].edge_labels]:
            r_r_wrong = r_r_wrong + 1
        elif 1 in [*test_Dataset[i].edge_labels]:
            w_r += 1
        else:
            w_w += 1
        
    reaction_edges = torch.where(predicted_classes == 1)[0]
    print(f"Bond: {reaction_edges.numpy()}")

    print("\nAll bonds:")
    for edge_idx in reaction_edges:
        src_atom = test_Dataset[i].edge_index[0, edge_idx].item()
        dst_atom = test_Dataset[i].edge_index[1, edge_idx].item()
        probability = edge_probs[edge_idx, 1].item()
        print(f"  Bond {edge_idx}: Atom {src_atom} - Atom {dst_atom}, Possibility: {probability:.3f}")
    
    print()

print('All datas:', all_sets)
print('Correct:', correct_sets)
print('Accuracy:', correct_sets / all_sets)
print()
print()
print('{0} {1} {2}'.format(r_r, r_r_wrong, w_r))
print('{0} {1}'.format(w_w, r_w))

Bonds: [array([[1.5347515 , 0.        , 1.125787  , 0.85522735, 0.        ],
       [1.5347515 , 0.        , 1.125787  , 0.85522735, 0.        ],
       [1.521402  , 0.        , 1.0114858 , 1.0748427 , 0.        ],
       [1.521402  , 0.        , 1.0114858 , 1.0748427 , 0.        ],
       [1.5160273 , 0.        , 0.97942656, 1.0465547 , 0.        ],
       [1.5160273 , 0.        , 0.97942656, 1.0465547 , 0.        ],
       [1.5149261 , 0.        , 1.0975404 , 0.93185246, 0.        ],
       [1.5149261 , 0.        , 1.0975404 , 0.93185246, 0.        ],
       [1.5085087 , 0.        , 1.2741531 , 1.0452224 , 0.        ],
       [1.5085087 , 0.        , 1.2741531 , 1.0452224 , 0.        ],
       [1.5082477 , 0.        , 1.2735482 , 1.0452974 , 0.        ],
       [1.5082477 , 0.        , 1.2735482 , 1.0452974 , 0.        ],
       [1.4365118 , 1.        , 0.9800529 , 0.8842155 , 0.        ],
       [1.4365118 , 1.        , 0.9800529 , 0.8842155 , 0.        ],
       [1.4353892 , 1.    

In [ ]:
0.83206, 0.82443, 0.85606, 0.85496, 0.80303

In [ ]:
0.86260, 0.82443, 0.81818, 0.80916, 0.82576